In [0]:
# ============================================================
# NOTEBOOK : DIM_CUSTOMER
# PURPOSE  : CUSTOMER SCD2 INCREMENTAL LOAD
# ============================================================

# ============================================================
# IMPORT PACKAGES
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid

In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

Max Date updated successfully


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER

FN_LOGGER LOADED SUCCESSFULLY


In [0]:
try:

    metadata = get_metadata("customer_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.dim_customer"

    pipeline_name = "PL_DIM_CUSTOMER"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            customer_id,
            first_name,
            last_name,
            email,
            city,
            modified_date

        FROM bronze.customer_tbl

        WHERE modified_date > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_customer"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(rows_read)

except Exception as e:

    print(f"Bronze View is not created : {table_name}")

    raise(e)

Last Watermark : 2025-05-25 12:00:00
Bronze View Created : customer_tbl
0


In [0]:
try:

    # ========================================================
    # CREATE SILVER TEMP VIEW
    # ========================================================

    silver_df = spark.sql("""

        SELECT

            customer_id,
            first_name,
            last_name,

            concat(first_name,' ',last_name)
                AS customer_name,

            email,
            city,
            modified_date,

            sha2(
                concat_ws(
                    '|',
                    concat(first_name,' ',last_name),
                    email,
                    city
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_customer

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_customer"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)

Silver View Created : customer_tbl


In [0]:
try:

    # ========================================================
    # MERGE LOGIC
    # ========================================================

    spark.sql("""

        MERGE INTO silver.dim_customer AS target

        USING vw_silver_customer AS source

        ON target.customer_id = source.customer_id
           AND target.is_current = 1

        WHEN MATCHED
             AND target.hash_key <> source.hash_key

        THEN UPDATE SET

            target.effective_end_date = current_timestamp(),
            target.is_current = 0

        WHEN NOT MATCHED

        THEN INSERT
        (
            customer_id,
            first_name,
            last_name,
            customer_name,
            email,
            city,
            modified_date,
            hash_key,
            effective_start_date,
            effective_end_date,
            is_current,
            is_deleted
        )

        VALUES
        (
            source.customer_id,
            source.first_name,
            source.last_name,
            source.customer_name,
            source.email,
            source.city,
            source.modified_date,
            source.hash_key,
            source.effective_start_date,
            source.effective_end_date,
            source.is_current,
            source.is_deleted
        )

    """)

    print(f"Merge Completed : {table_name}")

    # ========================================================
    # SOFT DELETE LOGIC
    # ========================================================

    spark.sql("""

        UPDATE silver.dim_customer

        SET

            is_deleted = 1,
            is_current = 0,
            effective_end_date = current_timestamp()

        WHERE customer_id NOT IN
        (
            SELECT customer_id
            FROM vw_silver_customer
        )

        AND is_current = 1

    """)

    print(f"Soft Delete Completed : {table_name}")

except Exception as e:

    print(f"Merge Failed : {table_name}")

    raise(e)

In [0]:
try:

    # ========================================================
    # GET MAX DATE
    # ========================================================

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    # ========================================================
    # UPDATE WATERMARK
    # ========================================================

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    # ========================================================
    # ROWS WRITTEN
    # ========================================================

    rows_written = silver_df.count()

    # ========================================================
    # END TIME
    # ========================================================

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    # ========================================================
    # INSERT AUDIT LOG
    # ========================================================

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"Function updation is successed : {table_name}")

except Exception as e:

    # ========================================================
    # INSERT ERROR LOG
    # ========================================================

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "DIM_CUSTOMER",
        str(e)

    )

    print(f"Function updations is failed : {table_name}")

    raise(e)

No Incremental Records Found
Audit Log Inserted : customer_tbl
Function updation is successed : customer_tbl
